In [1]:
import os
import requests
import numpy as np
import matplotlib.pyplot as plt
import glob
from iminuit import Minuit
from iminuit.cost import LeastSquares
from scipy.stats import chi2
import scipy as s
import scipy.stats as sc
from scipy.stats import chi2 as chi2_dist
from scipy.stats import t,norm
plt.rcParams['text.usetex'] = False
try:
    import Uomo_lucertola_vecchia as lib2
    print('Successo Importazione')
except ModuleNotFoundError:
    print(f"ERRORE: Impossibile trovare il modulo Uomo_lucertola. Assicurati che 'Uomo_lucertola.py' sia nella stessa cartella del notebook: {os.getcwd()}")
try:
    import Uomo_lucertola as lib
    print('Successo Importazione 2')
except ModuleNotFoundError:
    print(f"ERRORE: Impossibile trovare il modulo Uomo_lucertola2. Assicurati che 'Uomo_lucertola2.py' sia nella stessa cartella del notebook: {os.getcwd()}")


import uncertainties
from uncertainties import ufloat, correlated_values
from uncertainties import ufloat
from uncertainties import unumpy
cartella_dati = "misure"


ERRORE: Impossibile trovare il modulo Uomo_lucertola. Assicurati che 'Uomo_lucertola.py' sia nella stessa cartella del notebook: c:\Users\Andrea\Desktop\lab nucleare
Successo Importazione 2


Adesso stiamo calcolando il rate. Che si calcola. Il rate è l'area del picco fratto il tempo. 

#  Rate Rivelatore da 2''

In [2]:
file_rate_list = ["rategrande_1.dat", "rategrande_2.dat", "rategrande_3.dat"]
#c_v_max = 3500; c_v_min = 2700
c_v_max = 3400; c_v_min = 2800
numeri = np.array([1, 2, 3])
risultati_rate = []
valori_rate_ufloat = []
tempo = 600 #10 minuti = 600 secondi

for filename, numero in zip(file_rate_list, numeri):
    print(f"\n{'='*60}")
    print(f"  Misura Numero = {numero} (Taglio tra {c_v_min} e {c_v_max})")
    print(f"{'='*60}")
    
    percorso_completo = os.path.join(cartella_dati, filename)
    canali, conteggi = lib.estrai_spettro(percorso_completo, show_plot=False)
    canali, conteggi = lib.rebin(canali, conteggi, n=4)

    # 3. Ora c_min e c_max sono numeri singoli esatti per questo specifico file
    mask = (canali >= c_v_min) & (canali <= c_v_max)
    x_picco = canali[mask]
    y_picco = conteggi[mask]
    
    # Stime iniziali
    mu_guess = x_picco[np.argmax(y_picco)]
    area_guess = np.sum(y_picco)
    fondo_guess = np.mean(y_picco[:5])
    sigma_guess = 15
    x0_guess = x_picco[0]
    c_guess = np.mean(y_picco[-5:])
    if c_guess <= 0: c_guess = 1.0

    # 3. Stima dell'altezza dell'esponenziale 'A_fondo' (coda sinistra)
    sinistra_mean = np.mean(y_picco[:5])
    A_fondo_guess = sinistra_mean - c_guess
    if A_fondo_guess <= 0: A_fondo_guess = 10.0 # Sicurezza

    # 4. Stima di 'tau' (un terzo della finestra)
    tau_guess = (x_picco[-1] - x_picco[0]) / 3.0

    # 5. Stime del picco (mu e sigma)
    mu_guess = x_picco[np.argmax(y_picco)]
    sigma_guess = 15.0 # Per i NaI di solito è un valore iniziale eccellente

    # 6. Stima dell'Area (Totale - Area del fondo a trapezio)
    fondo_medio = (sinistra_mean + c_guess) / 2.0
    area_guess = np.sum(y_picco) - (fondo_medio * len(x_picco))
    if area_guess <= 0: area_guess = np.sum(y_picco) * 0.5
    
    # 4. Corretti i parametri per la parabola (a, b, c)
    parametri_iniziali = {
        'Area': area_guess, 
        'mu': mu_guess, 
        'sigma': sigma_guess, 
        'A_fondo': A_fondo_guess,
        'tau': tau_guess,
        'c': fondo_guess,
        'x0': x0_guess
    }
    # 5. Corretti gli assegnamenti di CDF e PDF
    fit = lib.FitLikelihoodBomberone(
        canali=x_picco,
        conteggi=y_picco,
        modello_cdf=lib.picco_esponenziale_cdf,
        modello_pdf=lib.picco_esponenziale_pdf,
        initial_params=parametri_iniziali,
        title=f"Na-22 @ {numero}"
    )
    
    fit.perform_fit(silent=True)
    #fit.plot_results(componenti=[('Gaussiana', lib.funzione_gaussiana_pdf), ('Fondo (esponenziale)', lib.funzione_esponenziale_pdf)])  
    
    if fit.is_fit_valid:
        mu_ufloat  = ufloat(fit.fit_result['mu'][0], fit.fit_result['mu'][1])
        sig_ufloat = ufloat(fit.fit_result['sigma'][0], fit.fit_result['sigma'][1])
        area_ufloat = ufloat(fit.fit_result['Area'][0], fit.fit_result['Area'][1])
        risultati_rate.append((numero, mu_ufloat, sig_ufloat, area_ufloat))
        print(f"  → Picco = {(mu_ufloat._nominal_value):.4f} ± {float(sig_ufloat.std_dev):.4f}")
        print(f"  → Sigma = {(sig_ufloat._nominal_value):.4f} ± {float(sig_ufloat.std_dev):.4f}")
        print(f"  → Area = {(area_ufloat._nominal_value):.4f} ± {float(area_ufloat.std_dev):.4f}")
    else:
        print(f"  ⚠️ Fit non valido per fit {numero}")
    rate_i = area_ufloat / tempo
    valori_rate_ufloat.append(rate_i)
    print(f"  → Rate = {rate_i:.4f} ± {rate_i.std_dev:.4f} counts/s")


  Misura Numero = 1 (Taglio tra 2800 e 3400)
  → Picco = 3105.0447 ± 2.2120
  → Sigma = 110.5059 ± 2.2120
  → Area = 21260.5485 ± 871.7895
  → Rate = 35.4342+/-1.4530 ± 1.4530 counts/s

  Misura Numero = 2 (Taglio tra 2800 e 3400)
  → Picco = 3105.8502 ± 2.1345
  → Sigma = 112.0406 ± 2.1345
  → Area = 21581.3528 ± 777.7301
  → Rate = 35.9689+/-1.2962 ± 1.2962 counts/s

  Misura Numero = 3 (Taglio tra 2800 e 3400)
  → Picco = 3092.2229 ± 2.3120
  → Sigma = 113.9771 ± 2.3120
  → Area = 21835.3210 ± 930.5443
  → Rate = 36.3922+/-1.5509 ± 1.5509 counts/s


In [3]:
valori = np.array([rate.n for rate in valori_rate_ufloat])
errori = np.array([rate.std_dev for rate in valori_rate_ufloat])

pesi = 1.0 / (errori**2)
rate_medio_pesato = np.sum(valori * pesi) / np.sum(pesi)
errore_medio_pesato = 1.0 / np.sqrt(np.sum(pesi))

# Crea il nuovo ufloat finale
rate_finale = ufloat(rate_medio_pesato, errore_medio_pesato)

print(f"\n--- RISULTATO FINALE ---")
print(f"Rate Medio del Rivelatore Grande: {rate_finale.n:.4f} ± {rate_finale.std_dev:.4f} cps")


--- RISULTATO FINALE ---
Rate Medio del Rivelatore Grande: 35.9169 ± 0.8207 cps


#  Rate Rivelatore 1''

In [4]:
file_rate_list2 = ["ratepiccolo_1.dat", "ratepiccolo_2.dat", "ratepiccolo_3.dat"]
c_v_max2 = 3500; c_v_min2 = 2700
#c_v_max2 = 3400; c_v_min2 = 2800
numeri = np.array([1, 2, 3])
risultati_rate2 = []
valori_rate_ufloat2 = []
tempo = 600 #10 minuti = 600 secondi

for filename, numero in zip(file_rate_list2, numeri):
    print(f"\n{'='*60}")
    print(f"  Misura Numero = {numero} (Taglio tra {c_v_min2} e {c_v_max2})")
    print(f"{'='*60}")
    
    percorso_completo = os.path.join(cartella_dati, filename)
    canali, conteggi = lib.estrai_spettro(percorso_completo, show_plot=False)
    canali, conteggi = lib.rebin(canali, conteggi, n=4)

    # 3. Ora c_min e c_max sono numeri singoli esatti per questo specifico file
    mask = (canali >= c_v_min2) & (canali <= c_v_max2)
    x_picco = canali[mask]
    y_picco = conteggi[mask]
    
    # Stime iniziali
    mu_guess = x_picco[np.argmax(y_picco)]
    area_guess = np.sum(y_picco)
    fondo_guess = np.mean(y_picco[:5])
    sigma_guess = 15
    x0_guess = x_picco[0]
    c_guess = np.mean(y_picco[-5:])
    if c_guess <= 0: c_guess = 1.0

    # 3. Stima dell'altezza dell'esponenziale 'A_fondo' (coda sinistra)
    sinistra_mean = np.mean(y_picco[:5])
    A_fondo_guess = sinistra_mean - c_guess
    if A_fondo_guess <= 0: A_fondo_guess = 10.0 # Sicurezza

    # 4. Stima di 'tau' (un terzo della finestra)
    tau_guess = (x_picco[-1] - x_picco[0]) / 3.0

    # 5. Stime del picco (mu e sigma)
    mu_guess = x_picco[np.argmax(y_picco)]
    sigma_guess = 15.0 # Per i NaI di solito è un valore iniziale eccellente

    # 6. Stima dell'Area (Totale - Area del fondo a trapezio)
    fondo_medio = (sinistra_mean + c_guess) / 2.0
    area_guess = np.sum(y_picco) - (fondo_medio * len(x_picco))
    if area_guess <= 0: area_guess = np.sum(y_picco) * 0.5
    
    # 4. Corretti i parametri per la parabola (a, b, c)
    parametri_iniziali = {
        'Area': area_guess, 
        'mu': mu_guess, 
        'sigma': sigma_guess, 
        'A_fondo': A_fondo_guess,
        'tau': tau_guess,
        'c': fondo_guess,
        'x0': x0_guess
    }
    # 5. Corretti gli assegnamenti di CDF e PDF
    fit = lib.FitLikelihoodBomberone(
        canali=x_picco,
        conteggi=y_picco,
        modello_cdf=lib.picco_esponenziale_cdf,
        modello_pdf=lib.picco_esponenziale_pdf,
        initial_params=parametri_iniziali,
        title=f"Na-22 @ {numero}"
    )
    
    fit.perform_fit(silent=True)
    #fit.plot_results(componenti=[('Gaussiana', lib.funzione_gaussiana_pdf), ('Fondo (esponenziale)', lib.funzione_esponenziale_pdf)])  
    
    if fit.is_fit_valid:
        mu_ufloat  = ufloat(fit.fit_result['mu'][0], fit.fit_result['mu'][1])
        sig_ufloat = ufloat(fit.fit_result['sigma'][0], fit.fit_result['sigma'][1])
        area_ufloat = ufloat(fit.fit_result['Area'][0], fit.fit_result['Area'][1])
        risultati_rate2.append((numero, mu_ufloat, sig_ufloat, area_ufloat))
        print(f"  → Picco = {(mu_ufloat._nominal_value):.4f} ± {float(sig_ufloat.std_dev):.4f}")
        print(f"  → Sigma = {(sig_ufloat._nominal_value):.4f} ± {float(sig_ufloat.std_dev):.4f}")
        print(f"  → Area = {(area_ufloat._nominal_value):.4f} ± {float(area_ufloat.std_dev):.4f}")
    else:
        print(f"  ⚠️ Fit non valido per fit {numero}")
    rate_i = area_ufloat / tempo
    valori_rate_ufloat2.append(rate_i)
    print(f"  → Rate = {rate_i:.4f} ± {rate_i.std_dev:.4f} counts/s")


  Misura Numero = 1 (Taglio tra 2700 e 3500)
  → Picco = 3098.2055 ± 5.4451
  → Sigma = 174.2020 ± 5.4451
  → Area = 9693.8806 ± 438.6732
  → Rate = 16.1565+/-0.7311 ± 0.7311 counts/s

  Misura Numero = 2 (Taglio tra 2700 e 3500)
  → Picco = 3087.8240 ± 6.0333
  → Sigma = 176.7328 ± 6.0333
  → Area = 9612.5698 ± 544.4080
  → Rate = 16.0209+/-0.9073 ± 0.9073 counts/s

  Misura Numero = 3 (Taglio tra 2700 e 3500)
  → Picco = 3088.8930 ± 5.1740
  → Sigma = 174.8151 ± 5.1740
  → Area = 9889.5028 ± 430.2199
  → Rate = 16.4825+/-0.7170 ± 0.7170 counts/s


In [5]:
valori2 = np.array([rate.n for rate in valori_rate_ufloat2])
errori2 = np.array([rate.std_dev for rate in valori_rate_ufloat2])

pesi2 = 1.0 / (errori2**2)
rate_medio_pesato2 = np.sum(valori2 * pesi2) / np.sum(pesi2)
errore_medio_pesato2 = 1.0 / np.sqrt(np.sum(pesi2))

# Crea il nuovo ufloat finale
rate_finale2 = ufloat(rate_medio_pesato2, errore_medio_pesato2)

print(f"\n--- RISULTATO FINALE ---")
print(f"Rate Medio del Rivelatore Piccolo: {rate_finale2.n:.4f} ± {rate_finale2.std_dev:.4f} cps")





--- RISULTATO FINALE ---
Rate Medio del Rivelatore Piccolo: 16.2498 ± 0.4459 cps


#  Rate Coincidenza

In [6]:
# Carica i due file (supponendo che ogni riga contenga un solo numero: i conteggi in quel canale)
conteggi1 = np.loadtxt("misure/ratetrigger_1h.dat")
conteggi2 = np.loadtxt("misure/ratetrigger_30m.dat")

# Verifica che i due spettri abbiano la stessa lunghezza (stesso numero di canali)
if len(conteggi1) != len(conteggi2):
    raise ValueError("I due spettri hanno numero di canali diverso!")

# Somma canale per canale
conteggi_totali = conteggi1 + conteggi2
# (Opzionale) Salva il risultato in un nuovo file .dat
np.savetxt("C:/Users/Andrea/Desktop/lab nucleare/misure/histo_somma.dat", conteggi_totali, fmt="%d")

In [7]:
file_rate_list3 = ["histo_somma.dat"]
c_v_max3 = 3500; c_v_min3 = 2700
#c_v_max3 = 3400; c_v_min3 = 2800
risultati_rate3 = []
valori_rate_ufloat3 = []
tempo3 = 1800 #10 minuti = 600 secondi

for filename in zip(file_rate_list3):
    print(f"\n{'='*60}")
    print(f"  (Taglio tra {c_v_min3} e {c_v_max3})")
    print(f"{'='*60}")
    
    percorso_completo = os.path.join(cartella_dati, "histo_somma.dat")
    canali, conteggi = lib.estrai_spettro(percorso_completo, show_plot=False)
    canali, conteggi = lib.rebin(canali, conteggi, n=4)

    # 3. Ora c_min e c_max sono numeri singoli esatti per questo specifico file
    mask = (canali >= c_v_min3) & (canali <= c_v_max3)
    x_picco = canali[mask]
    y_picco = conteggi[mask]
    
    # Stime iniziali
    mu_guess = x_picco[np.argmax(y_picco)]
    area_guess = np.sum(y_picco)
    fondo_guess = np.mean(y_picco[:5])
    sigma_guess = 15
    x0_guess = x_picco[0]
    c_guess = np.mean(y_picco[-5:])
    if c_guess <= 0: c_guess = 1.0

    # 3. Stima dell'altezza dell'esponenziale 'A_fondo' (coda sinistra)
    sinistra_mean = np.mean(y_picco[:5])
    A_fondo_guess = sinistra_mean - c_guess
    if A_fondo_guess <= 0: A_fondo_guess = 10.0 # Sicurezza

    # 4. Stima di 'tau' (un terzo della finestra)
    tau_guess = (x_picco[-1] - x_picco[0]) / 3.0

    # 5. Stime del picco (mu e sigma)
    mu_guess = x_picco[np.argmax(y_picco)]
    sigma_guess = 15.0 # Per i NaI di solito è un valore iniziale eccellente

    # 6. Stima dell'Area (Totale - Area del fondo a trapezio)
    fondo_medio = (sinistra_mean + c_guess) / 2.0
    area_guess = np.sum(y_picco) - (fondo_medio * len(x_picco))
    if area_guess <= 0: area_guess = np.sum(y_picco) * 0.5
    
    # 4. Corretti i parametri per la parabola (a, b, c)
    parametri_iniziali = {
        'Area': area_guess, 
        'mu': mu_guess, 
        'sigma': sigma_guess, 
        'A_fondo': A_fondo_guess,
        'tau': tau_guess,
        'c': fondo_guess,
        'x0': x0_guess
    }
    # 5. Corretti gli assegnamenti di CDF e PDF
    fit = lib.FitLikelihoodBomberone(
        canali=x_picco,
        conteggi=y_picco,
        modello_cdf=lib.picco_esponenziale_cdf,
        modello_pdf=lib.picco_esponenziale_pdf,
        initial_params=parametri_iniziali,
        title=f"Na-22 @ Somma set in Coincidenza"
    )
    
    fit.perform_fit()
    #fit.plot_results(componenti=[('Gaussiana', lib.funzione_gaussiana_pdf), ('Fondo (esponenziale)', lib.funzione_esponenziale_pdf)])  
    
    if fit.is_fit_valid:
        mu_ufloat3  = ufloat(fit.fit_result['mu'][0], fit.fit_result['mu'][1])
        sig_ufloat3 = ufloat(fit.fit_result['sigma'][0], fit.fit_result['sigma'][1])
        area_ufloat3 = ufloat(fit.fit_result['Area'][0], fit.fit_result['Area'][1])
        risultati_rate3.append((numero, mu_ufloat3, sig_ufloat3, area_ufloat3))
        print(f"  → Picco = {(mu_ufloat3._nominal_value):.4f} ± {float(sig_ufloat3.std_dev):.4f}")
        print(f"  → Sigma = {(sig_ufloat3._nominal_value):.4f} ± {float(sig_ufloat3.std_dev):.4f}")
        print(f"  → Area = {(area_ufloat3._nominal_value):.4f} ± {float(area_ufloat3.std_dev):.4f}")
    else:
        print(f"  ⚠️ Fit non valido")
    rate_i3 = area_ufloat3 / tempo3
    valori_rate_ufloat3.append(rate_i3)
    print(f"  → Rate = {rate_i3:.4f} ± {rate_i3.std_dev:.4f} counts/s")


  (Taglio tra 2700 e 3500)
 RISULTATI FIT LIKELIHOOD: Na-22 @ Somma set in Coincidenza
      Area =   1.19e+04 ± 3.456
        mu =       3039 ± 0.000127
     sigma =      111.8 ± 0.08602
   A_fondo =      3.216 ± 0.01202
       tau =       1200 ± 9.859
         c =     -2.325 ± 0.008063
        x0 =       2702 ± 27.02
--------------------------------------------------
Chi2 / ndof = 165.70 / 161 = 1.03
p-value     = 0.3833
  → Picco = 3039.2524 ± 0.0860
  → Sigma = 111.8422 ± 0.0860
  → Area = 11898.3034 ± 3.4564
  → Rate = 6.6102+/-0.0019 ± 0.0019 counts/s


In [8]:
valori3 = np.array([rate.n for rate in valori_rate_ufloat3])
errori3 = np.array([rate.std_dev for rate in valori_rate_ufloat3])

pesi3 = 1.0 / (errori3**2)
rate_medio_pesato3 = np.sum(valori3 * pesi3) / np.sum(pesi3)
errore_medio_pesato3 = 1.0 / np.sqrt(np.sum(pesi3))

# Crea il nuovo ufloat finale
rate_finale3 = ufloat(rate_medio_pesato3, errore_medio_pesato3)

print(f"\n--- RISULTATO FINALE ---")
print(f"Rate Medio della Coincidenza: {rate_finale3.n:.4f} ± {rate_finale3.std_dev:.4f} cps")





--- RISULTATO FINALE ---
Rate Medio della Coincidenza: 6.6102 ± 0.0019 cps


# Efficienze

rate_finale = rate rivelatore 2'' Grande
rate_finale2 = rate rivelatore 1'' Piccolo
rate_finale3 = rate coincidenza

In [9]:
ACTIVITY_BQ_vecchia  = (10*(10**-6))*(3.7*10**10)  # [Bq]   attività sorgente (tipica Na-22 da lab: 1–100 kBq)
ACTIVITY_BQ  = 18776  # [Bq]   attività sorgente (tipica Na-22 da lab: 1–100 kBq) --- IGNORE ---
epsilon_piccolo = rate_finale3 / rate_finale2
epsilon_grande = rate_finale3 / rate_finale

print(f"\n--- EFFICIENZA RIVELATORI ---")
print(f"Efficienza del Rivelatore Piccolo: {epsilon_piccolo.n:.4f} ± {epsilon_piccolo.std_dev:.4f}")
print(f"Efficienza del Rivelatore Grande: {epsilon_grande.n:.4f} ± {epsilon_grande.std_dev:.4f}")

BR = 0.905 * 2  #IL FATTORE 2 è MOLTO IMPORTANTE PERCHE' IL NA-22 EMETTE DUE FOTONI IN COINCIDENZA, NON UNO SOLO


--- EFFICIENZA RIVELATORI ---
Efficienza del Rivelatore Piccolo: 0.4068 ± 0.0112
Efficienza del Rivelatore Grande: 0.1840 ± 0.0042


In [10]:
angolo_solido = 4 * np.pi * (rate_finale*rate_finale2) / (2 * ACTIVITY_BQ * BR * rate_finale3) # [steradianti] angolo solido visto dalla sorgente
print(f"\n--- ANGOLO SOLIDO ---")
print(f"Angolo solido visto dalla sorgente: {angolo_solido.n:.4f} ± {angolo_solido.std_dev:.4f} sr")


--- ANGOLO SOLIDO ---
Angolo solido visto dalla sorgente: 0.0163 ± 0.0006 sr


In [11]:
print(f"\n--- ANGOLO SOLIDO ANALITICO (PER PUNTIFORME)---")
Ra = 0.0127
la = 0.18
cos_angolo_delta = la / np.sqrt((Ra**2) + (la**2))
angolo_solido_analitico = 2 * np.pi * (1 - cos_angolo_delta)
print(f"Angolo solido analiticamente: {angolo_solido_analitico:.4f}")


--- ANGOLO SOLIDO ANALITICO (PER PUNTIFORME)---
Angolo solido analiticamente: 0.0156


DA SIMULAZIONE MONTECARLO

Frazione di Angolo Solido (dOmega/4pi) = 1.239894e-03

 Angolo Solido (dOmega) = 0.015646 sr

In [12]:
ang_solido_anal = angolo_solido_analitico
ang_solido_sperimentale = angolo_solido.n
err_ang_solido_sperimentale = angolo_solido.std_dev
ang_solido_mc = 0.015646

t_mc = lib.test_compatibilita(ang_solido_mc, ang_solido_sperimentale, 0.0, err_ang_solido_sperimentale,
                                val1_name=f"t(Angolo Solido Montecarlo)", val2_name=f"t(Angolo Solido Sperimentale)", use_ttest= False)

t_anal = lib.test_compatibilita(ang_solido_anal, ang_solido_sperimentale, 0.0, err_ang_solido_sperimentale, 
                                val1_name=f"t(Angolo Solido Analitico)", val2_name=f"t(Angolo Solido Sperimentale)", use_ttest=False)



Test di compatibilità tra t(Angolo Solido Montecarlo) (1.565e-02 ± 0.00e+00) e t(Angolo Solido Sperimentale) (1.632e-02 ± 5.83e-04):
  Differenza: 6.78e-04
  Incertezza sulla differenza (denominatore di T): 5.83e-04
  Statistica del test (T o Z): 1.16
  Distribuzione usata: Normale Standard
  P-value (due code): 0.2446
  I due valori sono COMPATIBILI (p > 0.05).

Test di compatibilità tra t(Angolo Solido Analitico) (1.558e-02 ± 0.00e+00) e t(Angolo Solido Sperimentale) (1.632e-02 ± 5.83e-04):
  Differenza: 7.43e-04
  Incertezza sulla differenza (denominatore di T): 5.83e-04
  Statistica del test (T o Z): 1.28
  Distribuzione usata: Normale Standard
  P-value (due code): 0.2023
  I due valori sono COMPATIBILI (p > 0.05).
